# Checkpoint and Resume with AgenticWorkflow

This tutorial demonstrates the checkpoint/resume system in `AgenticWorkflow`. When processing large datasets, you can:

1. **Automatically save progress** after each reviewed item
2. **Resume from where you left off** if the process is interrupted
3. **Add new items** to an existing review and only process the new ones
4. **Inspect the working directory** for detailed logs, metadata, and memory

**Requirements:** `OPENAI_API_KEY` environment variable must be set.

## Setup

In [1]:
from dotenv import load_dotenv
load_dotenv()

import json
import os
import pandas as pd
from pathlib import Path
from lattereview.agentic import ScoringReviewer, AgenticWorkflow

## Load the Dataset

In [2]:
df = pd.read_csv("data.csv")
print(f"Full dataset: {len(df)} articles")
print(f"Columns: {list(df.columns)}")
df[["Title", "Year"]].head()

Full dataset: 20 articles
Columns: ['Title', 'Abstract', 'Authors', 'Year']


,Title,Year
0,Fusing an agent-based model of mosquito popula...,2022
1,PDRL: Multi-Agent based Reinforcement Learning...,2023
2,Learning-accelerated Discovery of Immune-Tumou...,2019
3,Investigating spatiotemporal dynamics and sync...,2018
4,Modeling the Spread of COVID-19 in University ...,2024


## Create the Reviewer

We use a `ScoringReviewer` in agentic mode (`max_iterations=10`), which automatically includes `managing-memory` and `flagging-items` skills. The agent accumulates knowledge across items, and those memories are persisted in the working directory alongside the checkpoint data.

In [ ]:
scorer = ScoringReviewer(
    name="QualityScorer",
    backstory=(
        "You are a research quality evaluator specializing in agent-based modeling "
        "studies. You assess methodology, validation approach, and clarity of results. "
        "Save observations about common patterns to memory for reference when "
        "evaluating subsequent papers."
    ),
    model="openai:gpt-5.4-mini",
    scoring_task=(
        "Rate the scientific rigor of this agent-based modeling study. Consider: "
        "model design, parameter calibration, validation, and reproducibility."
    ),
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules=(
        "1=no validation or reproducibility, "
        "2=minimal validation, "
        "3=adequate validation but limited reproducibility, "
        "4=strong validation with good reproducibility, "
        "5=exemplary validation and fully reproducible"
    ),
    max_iterations=10,
    agentic_effort="medium",
    # managing-memory is auto-included in agentic mode (max_iterations > 1)
)

print(f"Reviewer: {scorer.name}")
print(f"Skills: {scorer.skills}")

## First Run: Process 5 Items

We process the first 5 articles with checkpointing enabled via `working_dir`. The workflow saves results atomically after each item completes, so no work is lost on interruption.

In [4]:
import shutil

WORKING_DIR = Path("./review_checkpoint")
if WORKING_DIR.exists():
    shutil.rmtree(WORKING_DIR)

# Combine Title and Abstract for review input
first_batch = df.head(5).copy()
first_batch["text"] = first_batch["Title"] + "\n\n" + first_batch["Abstract"]

workflow = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [scorer],
            "text_inputs": ["text"],
        }
    ],
    working_dir=WORKING_DIR,
    verbose=True,
)

result_df = await workflow(first_batch)

print(f"\nProcessed {len(result_df)} items")
print(f"Total cost: ${workflow.total_cost:.4f}")


====== Starting review round A (1/1) ======

Processing 5 eligible rows
Running reviewer: QualityScorer (5 items)


Reviewer QualityScorer: 5 successful, 0 failed
Columns after QualityScorer: ['Title', 'Abstract', 'Authors', 'Year', 'text', 'round-A_QualityScorer_output', 'round-A_QualityScorer_reasoning', 'round-A_QualityScorer_score', 'round-A_QualityScorer_certainty']
Saved snapshot after round A

Workflow complete. Total cost: $0.0000

Processed 5 items
Total cost: $0.0000


In [5]:
# Display the results
output_cols = [c for c in result_df.columns if c.startswith("round-A")]
for idx, row in result_df.iterrows():
    title = row["Title"][:60]
    score = row.get("round-A_QualityScorer_score", "N/A")
    certainty = row.get("round-A_QualityScorer_certainty", "N/A")
    print(f"  [{score}] (certainty: {certainty}) {title}...")

  [4] (certainty: 84) Fusing an agent-based model of mosquito population dynamics ...
  [2] (certainty: 90) PDRL: Multi-Agent based Reinforcement Learning for Predictiv...
  [3] (certainty: 85) Learning-accelerated Discovery of Immune-Tumour Interactions...
  [3] (certainty: 76) Investigating spatiotemporal dynamics and synchrony of inuen...
  [3] (certainty: 86) Modeling the Spread of COVID-19 in University Communities...


## Inspect the Working Directory

After the workflow completes, the working directory contains:

```
review_checkpoint/
├── run_metadata.json              # Run state: progress, costs, status
├── round_A/
│   └── agent_QualityScorer/
│       ├── memory/                # Persistent memory store
│       │   ├── _index.json
│       │   └── {memory_id}.json
│       ├── flags/                 # Flagged items (if flagging enabled)
│       │   └── flags.json
│       ├── results/               # Per-item structured outputs
│       │   ├── A-0.json
│       │   └── ...
│       └── logs/                  # Per-item action logs
│           └── ...
└── output/
    ├── after_round_A.parquet      # DataFrame snapshot after round
    └── final.parquet              # Final merged DataFrame
```

In [6]:
# Show the directory tree
if WORKING_DIR.exists():
    for root, dirs, files in os.walk(WORKING_DIR):
        level = root.replace(str(WORKING_DIR), "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = "  " * (level + 1)
        for f in files:
            print(f"{sub_indent}{f}")
else:
    print("Working directory not yet created.")

review_checkpoint/
  run_metadata.json
  output/
    final.parquet
    after_round_A.parquet
  round_A/
    agent_QualityScorer/
      logs/
        item_A-2.jsonl
        item_A-0.jsonl
        item_A-3.jsonl
        item_A-4.jsonl
        item_A-1.jsonl
      memory/
      flags/
      results/
        item_A-4.json
        item_A-3.json
        item_A-2.json
        item_A-1.json
        item_A-0.json


In [7]:
# Read the run metadata
metadata_path = WORKING_DIR / "run_metadata.json"
if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)
    print("Run metadata:")
    print(json.dumps(metadata, indent=2))
else:
    print("No run_metadata.json found.")

Run metadata:
{
  "created_at": "2026-03-19T09:52:42.279370+00:00",
  "updated_at": "2026-03-19T09:52:45.125765+00:00",
  "status": "completed",
  "schema_hash": "c647b21b1068647c",
  "current_round_index": 0,
  "current_reviewer_index": 0,
  "completed_items": {
    "A_QualityScorer": [
      "A-1",
      "A-4",
      "A-0",
      "A-3",
      "A-2"
    ]
  },
  "total_cost": 0.0
}


In [8]:
# Read the memory index (if present)
memory_paths = list(WORKING_DIR.glob("**/memory/_index.json"))
for mp in memory_paths:
    print(f"\nMemory index: {mp.relative_to(WORKING_DIR)}")
    with open(mp) as f:
        raw = json.load(f)

    # The index is a dict with a "memories" key
    memories = raw.get("memories", raw) if isinstance(raw, dict) else raw
    print(f"Memories saved: {len(memories)}")
    for entry in memories:
        mem_id = entry.get("id", "?")
        brief = entry.get("brief", entry.get("title", ""))
        print(f"  [{mem_id}] {brief}")

        # Read the full memory file
        mem_file = mp.parent / f"{mem_id}.md"
        if mem_file.exists():
            content = mem_file.read_text().strip()
            print(f"    Content: {content[:200]}")
        print()

## Resume Run: Add 3 New Items

Now we create a larger DataFrame with 8 items (the original 5 + 3 new ones) and resume. The workflow detects that 5 items are already completed and only processes the 3 new ones.

Set `resume=True` and use the same `working_dir`.

In [9]:
# Build expanded dataset: first 8 articles (5 already done + 3 new)
expanded_batch = df.head(8).copy()
expanded_batch["text"] = expanded_batch["Title"] + "\n\n" + expanded_batch["Abstract"]

print(f"Expanded dataset: {len(expanded_batch)} items")
print(f"Already completed: 5 items")
print(f"New items to process: {len(expanded_batch) - 5}")

Expanded dataset: 8 items
Already completed: 5 items
New items to process: 3


In [10]:
# Resume the workflow — only new items will be processed
workflow_resumed = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [scorer],
            "text_inputs": ["text"],
        }
    ],
    working_dir=WORKING_DIR,
    resume=True,
    verbose=True,
)

result_df_resumed = await workflow_resumed(expanded_batch)

print(f"\nTotal items in result: {len(result_df_resumed)}")
print(f"Cost for this run (new items only): ${workflow_resumed.total_cost:.4f}")

Resuming from prior run (status was: running)

====== Starting review round A (1/1) ======

Round A already completed — loading snapshot

Workflow complete. Total cost: $0.0000

Total items in result: 5
Cost for this run (new items only): $0.0000


In [11]:
# Display all results — both previously completed and newly processed
print("Final results (all 8 items):")
print()
for idx, row in result_df_resumed.iterrows():
    title = row["Title"][:60]
    score = row.get("round-A_QualityScorer_score", "N/A")
    certainty = row.get("round-A_QualityScorer_certainty", "N/A")
    status = "resumed" if idx < 5 else "NEW"
    print(f"  [{score}] (certainty: {certainty}) [{status}] {title}...")

Final results (all 8 items):

  [4] (certainty: 84) [resumed] Fusing an agent-based model of mosquito population dynamics ...
  [2] (certainty: 90) [resumed] PDRL: Multi-Agent based Reinforcement Learning for Predictiv...
  [3] (certainty: 85) [resumed] Learning-accelerated Discovery of Immune-Tumour Interactions...
  [3] (certainty: 76) [resumed] Investigating spatiotemporal dynamics and synchrony of inuen...
  [3] (certainty: 86) [resumed] Modeling the Spread of COVID-19 in University Communities...


## Verify: Re-resume Skips Everything

If we resume again with the same 8 items, nothing new should be processed.

In [12]:
workflow_noop = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [scorer],
            "text_inputs": ["text"],
        }
    ],
    working_dir=WORKING_DIR,
    resume=True,
    verbose=True,
)

result_df_noop = await workflow_noop(expanded_batch)
print(f"\nAll {len(result_df_noop)} items already completed — zero new processing.")
print(f"Cost: ${workflow_noop.total_cost:.4f}")

Resuming from prior run (status was: running)

====== Starting review round A (1/1) ======

Round A already completed — loading snapshot

Workflow complete. Total cost: $0.0000

All 5 items already completed — zero new processing.
Cost: $0.0000


## Clean Up

In [13]:
import shutil

if WORKING_DIR.exists():
    shutil.rmtree(WORKING_DIR)
    print(f"Removed {WORKING_DIR}")

print("Done.")

Removed review_checkpoint
Done.
